# Bước 1: Tiền xử lý & RFM (`customer_data.csv`)

Luồng **giống** `01_DataProcessing.ipynb` (Online Retail), chỉ **nguồn dữ liệu** là bảng khách **một dòng / khách** (`customer_data.csv`).

## Quy ước RFM (đồng bộ với pipeline cũ)

| Chỉ số | Cách lấy từ `customer_data.csv` | Ghi chú |
|--------|----------------------------------|--------|
| **R** | `Last_Purchase_Recency` (ngày) | **R càng nhỏ** → mua **gần đây** hơn (cùng hướng diễn giải với NB Online Retail). |
| **F** | `Web_Purchases` + `Catalog_Purchases` + `Store_Purchases` + `Deals_Purchased` | Tổng số lần mua / đợt mua qua các kênh (proxy cho Frequency). |
| **M** | `Total_Amount_Spent` | Giá trị chi tiêu tích lũy. |

## Chuẩn hóa (giống NB01)

- $F' = \log(1+F)$, $M' = \log(1+M)$; **R** không log.
- **StandardScaler** trên $(R, F', M')$ → `R_z`, `F_z`, `M_z`.

**Output:** thư mục `data_customer/` — `rfm_customers.csv` (không ghi đè `data/rfm_customers.csv` của Online Retail).


In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

def _project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "requirements.txt").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path.cwd().resolve()


ROOT = _project_root()
DATA_PATH = ROOT / "customer_data.csv"
OUT_DIR = ROOT / "data_customer"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [2]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = [c.strip() for c in df.columns]
print("Raw shape:", df.shape)
print(df.head(3))


Raw shape: (10000, 42)
   Customer_ID  Birth_Year Education_Level Marital_Status  Annual_Income  \
0            1        1978      Graduation          Widow       40175.23   
1            2        1991             PhD          Widow       27979.68   
2            3        1968             PhD        Married       64747.68   

   Kids_Home  Teens_Home Customer_Since  Last_Purchase_Recency  Spent_Wine  \
0          1           1     26-09-2010                     68         440   
1          0           1     03-11-2012                     99         826   
2          0           0     31-10-2013                     63         895   

   ...  Mobile_App_User  Email_Subscriber  Device_Type  Referred_By_Friend  \
0  ...                1                 0      Desktop                   1   
1  ...                1                 1      Desktop                   1   
2  ...                0                 0      Desktop                   0   

   Social_Media_User  Membership_Tier  Total_A

In [3]:
# Một dòng / khách — không cần gom groupby như Online Retail
rfm = pd.DataFrame(
    {
        "customer_id": df["Customer_ID"].astype(int),
        "R": df["Last_Purchase_Recency"].astype(int),
        "F": (
            df["Web_Purchases"].astype(int)
            + df["Catalog_Purchases"].astype(int)
            + df["Store_Purchases"].astype(int)
            + df["Deals_Purchased"].astype(int)
        ),
        "M": df["Total_Amount_Spent"].astype(float),
    }
)

# Đảm bảo F > 0 cho log1p và assert giống NB01
if (rfm["F"] <= 0).any():
    rfm.loc[rfm["F"] <= 0, "F"] = 1

print(rfm.describe())
rfm.head(10)


       customer_id             R             F             M
count  10000.00000  10000.000000  10000.000000  10000.000000
mean    5000.50000     49.592500     17.970200   1447.778100
std     2886.89568     29.101686      5.709376    408.159165
min        1.00000      0.000000      1.000000    210.000000
25%     2500.75000     25.000000     14.000000   1158.750000
50%     5000.50000     49.000000     18.000000   1447.000000
75%     7500.25000     75.000000     22.000000   1736.000000
max    10000.00000     99.000000     35.000000   2684.000000


,customer_id,R,F,M
0,1,68,12,1265.0
1,2,99,12,1647.0
2,3,63,19,1806.0
3,4,65,22,1207.0
4,5,49,17,934.0
5,6,71,13,2436.0
6,7,46,18,1413.0
7,8,67,5,1851.0
8,9,12,21,1576.0
9,10,13,21,1228.0


In [4]:
rfm["F_log1p"] = np.log1p(rfm["F"].astype(float))
rfm["M_log1p"] = np.log1p(rfm["M"].astype(float))

feat_cols = ["R", "F_log1p", "M_log1p"]
scaler = StandardScaler()
Z = scaler.fit_transform(rfm[feat_cols])
rfm["R_z"] = Z[:, 0]
rfm["F_z"] = Z[:, 1]
rfm["M_z"] = Z[:, 2]

out_path = OUT_DIR / "rfm_customers.csv"
rfm.to_csv(out_path, index=False)
print("Saved:", out_path.resolve())
print("Rows:", len(rfm))
rfm.head()


Saved: /Users/kotori/GMS_AFKMC2/data_customer/rfm_customers.csv
Rows: 10000


,customer_id,R,F,M,F_log1p,M_log1p,R_z,F_z,M_z
0,1,68,12,1265.0,2.564949,7.143618,0.632555,-0.952687,-0.289916
1,2,99,12,1647.0,2.564949,7.407318,1.697839,-0.952687,0.557281
2,3,63,19,1806.0,2.995732,7.499423,0.460735,0.308290,0.853192
3,4,65,22,1207.0,3.135494,7.096721,0.529463,0.717398,-0.440581
4,5,49,17,934.0,2.890372,6.840547,-0.020361,-0.000119,-1.263602


In [5]:
assert rfm["R"].min() >= 0
assert rfm[["F", "M"]].min().min() > 0
print("Số khách hàng sau RFM:", len(rfm))
print("Z-score — mean ~ 0, std ~ 1:", rfm[["R_z", "F_z", "M_z"]].mean().round(4).to_dict())
print("Z-score std:", rfm[["R_z", "F_z", "M_z"]].std().round(4).to_dict())


Số khách hàng sau RFM: 10000
Z-score — mean ~ 0, std ~ 1: {'R_z': -0.0, 'F_z': 0.0, 'M_z': 0.0}
Z-score std: {'R_z': 1.0001, 'F_z': 1.0001, 'M_z': 1.0001}
